# C3-14 · Step 2: merge the LoRA adapter into Mistral 7B and export a GGUF for Ollama
Student: Shathursini J (IT23164062) · Component C3

**Why this notebook:** Ollama 0.34.1+ no longer accepts `ADAPTER` in a Modelfile ("LoRA adapters are no longer supported").
So the trained adapter (from notebook 02) is merged into the base model here, converted to GGUF and quantised to Q4_K_M,
the same size class as the base `mistral:7b-instruct-q4_K_M` used in the C3-13 evaluation.

**Runtime:** no GPU needed. Runtime > Change runtime type > **CPU** > Save (saves your GPU quota). Then Runtime > Run all.
Needs about 35 GB of free Colab disk and about 4.5 GB free in Google Drive for the result.

| Cell | What | Time (CPU runtime) |
|---|---|---|
| 1 | Drive + checks | ~1 min |
| 2 | download base Mistral 7B (14.5 GB) | ~5-10 min |
| 3 | merge W + (alpha/r)·B·A shard by shard | ~5 min |
| 4 | convert to GGUF f16 (llama.cpp) | ~10 min |
| 5 | build llama-quantize, quantise to Q4_K_M, save to Drive | ~20-30 min |


In [ ]:
# 1. Setup: Google Drive, paths, checks
import os, sys, glob, json, re, shutil, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK, ROOT = "/content/drive/MyDrive/c3_finetune", "/content"
else:                                              # offline dry-run only
    WORK, ROOT = os.environ["C3_WORK"], os.environ["C3_ROOT"]
MODEL_ID = os.environ.get("C3_MODEL", "unsloth/mistral-7b-instruct-v0.3")   # same base the adapter was trained on
ADAPTER = f"{WORK}/c3-mistral-lora"
BASE, MERGED, LLAMA = f"{ROOT}/base", f"{ROOT}/merged", f"{ROOT}/llama.cpp"
F16, Q4 = f"{ROOT}/c3-mistral-ft-f16.gguf", f"{ROOT}/c3-mistral-ft-Q4_K_M.gguf"

if not os.path.exists(f"{ADAPTER}/adapter_model.safetensors") and IN_COLAB:
    from google.colab import files                # adapter not in Drive -> upload c3-mistral-lora.zip
    print("Select c3-mistral-lora.zip")
    for name, data in files.upload().items():
        open(f"{ROOT}/{name}", "wb").write(data)
        shutil.unpack_archive(f"{ROOT}/{name}", f"{ROOT}/c3-mistral-lora")
    ADAPTER = f"{ROOT}/c3-mistral-lora"
assert os.path.exists(f"{ADAPTER}/adapter_model.safetensors"), f"adapter_model.safetensors not found in {ADAPTER}"
cfg = json.load(open(f"{ADAPTER}/adapter_config.json"))
print("adapter:", ADAPTER, "| base:", cfg["base_model_name_or_path"], "| r =", cfg["r"], "alpha =", cfg["lora_alpha"])
free = shutil.disk_usage(ROOT).free / 1e9
print(f"free Colab disk: {free:.0f} GB")
assert free > 35 or not IN_COLAB, "Need about 35 GB free disk: Runtime > Disconnect and delete runtime, then Run all again"


In [ ]:
# 2. Download the base model weights (bf16 safetensors + tokenizer)
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "safetensors", "sentencepiece", "protobuf"], check=True)
from huggingface_hub import snapshot_download
if not any(os.path.exists(p) for p in (f"{MERGED}/config.json", F16, Q4)):   # skip when already merged
    snapshot_download(MODEL_ID, local_dir=BASE, allow_patterns=["*.safetensors", "*.json", "tokenizer.model"])
    print(sorted(os.listdir(BASE)))
else:
    print("merged model already present - download skipped")


In [ ]:
# 3. Merge: W_merged = W + (alpha / r) * B @ A, one shard at a time (low RAM, no GPU)
import torch
from safetensors import safe_open
from safetensors.torch import load_file, save_file

def merge_lora(base_dir, adapter_dir, out_dir):
    cfg = json.load(open(f"{adapter_dir}/adapter_config.json"))
    scale = cfg["lora_alpha"] / cfg["r"]
    lora = {}
    with safe_open(f"{adapter_dir}/adapter_model.safetensors", "pt") as f:
        for k in f.keys():
            m = re.fullmatch(r"base_model\.model\.(.+)\.lora_([AB])(?:\.default)?\.weight", k)
            assert m, f"unexpected adapter tensor {k}"
            lora.setdefault(m.group(1) + ".weight", {})[m.group(2)] = f.get_tensor(k).float()
    os.makedirs(out_dir, exist_ok=True)
    shards = sorted(glob.glob(f"{base_dir}/*.safetensors"))
    assert shards, f"no .safetensors files in {base_dir}"
    done = set()
    for shard in shards:
        w = load_file(shard)
        for name in list(w):
            if name in lora:
                delta = (lora[name]["B"] @ lora[name]["A"]) * scale
                assert delta.shape == w[name].shape, (name, delta.shape, w[name].shape)
                w[name] = (w[name].float() + delta).to(w[name].dtype).contiguous()
                done.add(name)
        save_file(w, f"{out_dir}/{os.path.basename(shard)}", metadata={"format": "pt"})
        print("merged", os.path.basename(shard), "-", len(done), "of", len(lora), "LoRA layers so far")
        del w
    missing = set(lora) - done
    assert not missing, f"LoRA layers not found in the base model: {sorted(missing)[:5]}"
    for p in glob.glob(f"{base_dir}/*"):             # config, tokenizer, index files
        if os.path.isfile(p) and not p.endswith(".safetensors"):
            shutil.copy(p, out_dir)
    return len(done)

if not any(os.path.exists(p) for p in (f"{MERGED}/config.json", F16, Q4)):   # config.json is copied last
    n = merge_lora(BASE, ADAPTER, MERGED)
    layers = json.load(open(f"{MERGED}/config.json"))["num_hidden_layers"]
    print(f"merged {n} LoRA layers ({layers} transformer blocks x {n // layers} projections)")
    shutil.rmtree(BASE, ignore_errors=True)          # free 14.5 GB of disk
print(sorted(os.listdir(MERGED)) if os.path.isdir(MERGED) else "merge already done")


In [ ]:
# 4. Convert the merged model to GGUF (f16) with llama.cpp
if not os.path.isdir(LLAMA):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/ggml-org/llama.cpp", LLAMA], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"{LLAMA}/gguf-py"], check=True)
if not os.path.exists(Q4) and not os.path.exists(F16):
    subprocess.run([sys.executable, f"{LLAMA}/convert_hf_to_gguf.py", MERGED, "--outtype", "f16", "--outfile", F16], check=True)
if os.path.exists(F16):
    print(f"{os.path.getsize(F16) / 1e9:.1f} GB  {F16}")
    shutil.rmtree(MERGED, ignore_errors=True)        # free 14.5 GB of disk


In [ ]:
# 5. Quantise to Q4_K_M (same size class as the base model in Ollama), check it, save to Drive
QUANT = f"{LLAMA}/build/bin/llama-quantize"
if not os.path.exists(Q4):
    if not os.path.exists(QUANT):
        subprocess.run(["cmake", "-S", LLAMA, "-B", f"{LLAMA}/build", "-DGGML_CUDA=OFF", "-DLLAMA_CURL=OFF",
                        "-DCMAKE_BUILD_TYPE=Release"], check=True, stdout=subprocess.DEVNULL)
        subprocess.run(["cmake", "--build", f"{LLAMA}/build", "--config", "Release", "-j", str(os.cpu_count()),
                        "--target", "llama-quantize"], check=True, stdout=subprocess.DEVNULL)
    subprocess.run([QUANT, F16, Q4, "Q4_K_M"], check=True, stdout=subprocess.DEVNULL)
    os.remove(F16)

from gguf import GGUFReader
r = GGUFReader(Q4)
arch = bytes(r.fields["general.architecture"].parts[-1]).decode()
print(f"{os.path.getsize(Q4) / 1e9:.2f} GB | architecture {arch} | {len(r.tensors)} tensors")
assert arch == "llama" and len(r.tensors) > 0

target = f"{WORK}/c3-mistral-ft-Q4_K_M.gguf"
shutil.copy(Q4, target)
print("saved to Google Drive:", target)
print("Next: download it from Drive (My Drive/c3_finetune) into services/lead-intel/models/ on your laptop")
